# Word embeddings

Dans ce TP, nous utilisons les word embeddings pré-entraînés de [spaCy](https://spacy.io/), puis nous entraînons les nôtres sur un corpus de messages : un Skip-Gram avec échantillonnage négatif, écrit en [PyTorch](https://pytorch.org/).

In [ ]:
import collections
import itertools
import pathlib
import re
import time

import numpy as np
import spacy
import torch

## Modèles pré-entraînés avec spaCy

Pour utiliser un modèle de langue de spaCy, il faut d'abord le télécharger (voir les [modèles de spaCy](https://spacy.io/usage/models)). Tous ne contiennent pas de word embeddings : pour le français, `fr_core_news_sm` et `fr_dep_news_trf` n'en ont pas.

Nous travaillons avec `fr_core_news_lg` (545 Mo), qui relie 500 000 mots à autant de vecteurs de 300 dimensions, rangés du plus fréquent au plus rare. La casse compte : `Berlin` et `berlin` ont chacun leur vecteur.

In [ ]:
!python -m spacy download fr_core_news_lg

## Utilisation

Une fois le modèle chargé, `nlp` traite un texte et rend un `Doc`, dont chaque token a son vecteur :

In [ ]:
# Chargement du modèle
nlp = spacy.load("fr_core_news_lg")

# Traitement du texte par spaCy
doc = nlp("Ceci est un texte que je traite avec spaCy")

print("Vecteur du 4e token :")
print(doc[3].vector)

# Le vecteur moyen de la phrase : une représentation possible pour classer des
# textes, mais grossière (l'ordre des mots est perdu). Préférez un modèle
# d'embeddings de phrases (Sentence Transformers)
print("Vecteur moyen :")
print(doc.vector)

## Une table d'embeddings

Les vecteurs du modèle sont dans `nlp.vocab.vectors`, une table d'une ligne par mot. La classe `Embeddings` ci-dessous les range pour le reste du TP : `mot in modele` dit si le modèle connaît `mot`, `modele[mot]` rend son vecteur, et `modele.voisins(vecteur)` les mots les plus proches de `vecteur`, des couples `(mot, cosinus)`. Elle divise chaque ligne par sa norme : le produit scalaire de deux lignes est alors leur cosinus, et `voisins` compare un vecteur à tous les mots en une seule multiplication.

Ces vecteurs ont appris sur des pages web : leur vocabulaire compte aussi des graphies rares ou fautives (`mèdecin`, `linfirmière`), qui viennent parfois parmi les voisins.

In [ ]:
class Embeddings:
  """Une table d'embeddings : un vecteur de longueur 1 par mot."""

  def __init__(self, lignes, table):
    self.lignes = lignes  # mot -> numéro de sa ligne dans la table
    self.mots = {}  # numéro de ligne -> mot (en minuscules s'il y en a plusieurs)
    for mot, ligne in lignes.items():
      if ligne not in self.mots or mot.islower():
        self.mots[ligne] = mot
    # Lignes de longueur 1 : le produit scalaire de deux lignes est leur cosinus
    self.table = table / np.linalg.norm(table, axis=1, keepdims=True)

  def __contains__(self, mot):
    return mot in self.lignes

  def __getitem__(self, mot):
    return self.table[self.lignes[mot]]

  def voisins(self, vecteur, n=5):
    """Les n mots les plus proches de `vecteur` : des couples (mot, cosinus)."""
    cosinus = self.table @ (vecteur / np.linalg.norm(vecteur))
    return [
      (self.mots[i], round(float(cosinus[i]), 2)) for i in np.argsort(-cosinus)[:n]
    ]


# Les vecteurs pré-entraînés de spaCy
vecteurs = nlp.vocab.vectors
modele = Embeddings(
  {nlp.vocab.strings[cle]: ligne for cle, ligne in vecteurs.key2row.items()},
  vecteurs.data,
)
print(len(modele.lignes), "mots,", modele.table.shape)

In [ ]:
vecteur = modele["facile"]
vecteur.shape

In [ ]:
modele.voisins(modele["Berlin"])

In [ ]:
# Le produit scalaire de deux vecteurs de longueur 1 : leur similarité cosinus
print(modele["Berlin"] @ modele["Munich"])
print(modele["Berlin"] @ modele["Paris"])

## 1. Un prétraitement cohérent avec celui du modèle

`modele` ne connaît que les mots de son vocabulaire, écrits et découpés comme dans les textes où il a appris : un mot absent n'a pas de vecteur, et `modele[mot]` lève alors `KeyError`.

Découpez `phrase` en mots avec `split`, puis affichez les mots que le modèle connaît, et ceux qu'il ne connaît pas. Recommencez avec les tokens de `nlp(phrase)`. Quel découpage le modèle attend-il ? Affichez enfin les voisins de `Berlin` et ceux de `berlin` : la casse compte-t-elle pour ce modèle ? Qu'en déduire sur le prétraitement de vos propres textes ?

**Documentation utile**

- [`str.split`](https://docs.python.org/3/library/stdtypes.html#str.split)
- [`Token.text`](https://spacy.io/api/token#attributes)

In [ ]:
phrase = "Berlin et Munich sont deux grandes villes d'Allemagne"

# Votre code ici

### Conseils supplémentaires

Une liste en compréhension garde les mots connus : `[mot for mot in mots if mot in modele]`. Le texte d'un token de `nlp(phrase)` est `token.text`. Pour vérifier : `split` laisse `d'Allemagne`, que le modèle ne connaît pas, et spaCy en fait deux tokens, `d'` et `Allemagne`.

```python
mots = phrase.split()
print("Connus :", [mot for mot in mots if mot in modele])
print("Absents :", [mot for mot in mots if ...])

mots = [... for token in nlp(phrase)]
...

print(modele.voisins(modele["Berlin"]))
...
```

### Solution

In [ ]:
mots = phrase.split()
print("Connus :", [mot for mot in mots if mot in modele])
print("Absents :", [mot for mot in mots if mot not in modele])

# Le découpage de spaCy, proche de celui des textes où ces vecteurs ont appris :
# d'Allemagne devient d' et Allemagne, que le modèle connaît
mots = [token.text for token in nlp(phrase)]
print("Connus :", [mot for mot in mots if mot in modele])
print("Absents :", [mot for mot in mots if mot not in modele])

# La casse compte : Berlin et berlin ont chacun leur vecteur, et berlin, plus
# rare, a pour voisins d'autres mots en minuscules
print(modele.voisins(modele["Berlin"]))
print(modele.voisins(modele["berlin"]))

## 2. Analogies

La diapositive « Analogies » le montre : le décalage de *homme* à *femme* est le même que celui de *roi* à *reine*, et roi − homme + femme ≈ reine. D'où une question : `x` est à `y` ce que `z` est à quoi ? Le mot cherché est proche du vecteur `modele[y] - modele[x] + modele[z]`.

Calculez d'abord ce vecteur pour *homme*, *femme*, *roi*, et affichez ses plus proches voisins avec `modele.voisins`. Quel mot vient en tête, et pourquoi ? Écrivez ensuite `analogie(modele, x, y, z)` : comme `voisins`, elle renvoie la liste des couples `(mot, cosinus)` les plus proches de ce vecteur, le meilleur en tête, mais sans les mots de la question. Essayez-la sur *homme*, *femme*, *roi*, puis sur *roi*, *reine*, *oncle* : la réponse est-elle celle que vous attendiez ? La suite du TP appelle `analogie`.

In [ ]:
# Votre code ici : le décalage et modele.voisins


def analogie(modele, x, y, z):
  # Votre code ici
  ...

### Conseils supplémentaires

`modele["femme"] - modele["homme"] + modele["roi"]` est un vecteur comme un autre : `modele.voisins(vecteur)` rend ses 5 plus proches voisins, sans écarter aucun mot. Pour écarter les mots de la question, comparez les lignes : `modele.lignes[mot]` est la ligne de `mot` dans la table. Demandez quelques voisins de plus (`n=8`), retirez ceux dont la ligne est celle d'un mot de la question, et gardez les 5 premiers. Pour *homme*, *femme*, *roi*, `reine` vient en tête de `analogie`.

```python
vecteur = modele["femme"] - ... + ...
print(modele.voisins(vecteur))


def analogie(modele, x, y, z):
  vecteur = ...
  question = {modele.lignes[mot] for mot in (x, y, z)}
  proches = modele.voisins(vecteur, n=8)
  return [(mot, c) for mot, c in proches if ...][:5]


print(analogie(modele, "homme", "femme", "roi"))
print(analogie(modele, ...))
```

### Solution

In [ ]:
vecteur = modele["femme"] - modele["homme"] + modele["roi"]
# roi en tête : le vecteur reste proche de modele["roi"], et voisins n'écarte
# aucun mot
print(modele.voisins(vecteur))


def analogie(modele, x, y, z):
  # x est à y ce que z est à ? Les mots de la question sont écartés
  vecteur = modele[y] - modele[x] + modele[z]
  # Les lignes des mots de la question
  question = {modele.lignes[mot] for mot in (x, y, z)}
  proches = modele.voisins(vecteur, n=8)
  return [(mot, c) for mot, c in proches if modele.lignes[mot] not in question][:5]


print(analogie(modele, "homme", "femme", "roi"))
# Pas de tante parmi les cinq premiers : cousine, belle-sœur, nièce
print(analogie(modele, "roi", "reine", "oncle"))

## 3. Réussites, échecs et biais

Une analogie ne réussit pas toujours. Essayez `analogie` sur *France*, *Paris*, *Japon*, puis sur *France*, *Paris*, *Italie* : laquelle donne la capitale ? Que donne *homme*, *femme*, *médecin* ? Les vecteurs viennent de leur corpus d'entraînement : ce qu'ils associent, biais compris, c'est ce que ce corpus associe. Gardez les majuscules des noms propres (exercice 1).

In [ ]:
# Votre code ici

### Conseils supplémentaires

Un appel par triplet, dans l'ordre `x`, `y`, `z` de l'énoncé (*France* est à *Paris* ce que *Japon* est à ?). Pour *France*, *Paris*, *Japon*, `Tokyo` vient en tête.

```python
print(analogie(modele, "France", "Paris", "Japon"))
print(analogie(modele, ...))
print(analogie(modele, ...))
```

### Solution

In [ ]:
# Juste : Tokyo
print(analogie(modele, "France", "Paris", "Japon"))
# Faux : Bologne, Venise, Milan, et pas de Rome parmi les cinq premiers
print(analogie(modele, "France", "Paris", "Italie"))
# pédiatre, gynécologue, linfirmière (et mèdecin, une graphie fautive du web) :
# les associations du corpus
print(analogie(modele, "homme", "femme", "médecin"))

## Création de word embeddings

Apprendre des vecteurs sur vos propres données n'a rien de compliqué. C'est particulièrement utile sur un corpus spécialisé (actes notariés, langues anciennes…), dont les mots n'ont pas ailleurs le même sens, ou n'existent pas ailleurs.

Ces embeddings statiques, un vecteur par mot, s'entraînaient avec une bibliothèque dédiée, comme gensim ou fastText : c'est la manière de faire historique. Aujourd'hui, les représentations des mots s'apprennent à l'intérieur des modèles qui s'en servent (les transformers), et gensim est [en maintenance](https://github.com/piskvorky/gensim), sans nouvelle fonctionnalité. Nous écrivons donc le Skip-Gram nous-mêmes, en PyTorch, en une cinquantaine de lignes.

### Le corpus 20 Newsgroups

Ce corpus réunit environ 20 000 messages en anglais, répartis en 20 thèmes, puisqu'ils viennent de 20 newsgroups différents.

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz
!ls 20news-bydate-train/

In [ ]:
!ls 20news-bydate-test/

In [ ]:
# Un dossier par newsgroup, un message par fichier
contenus = [
  p.read_text(encoding="latin-1") for p in pathlib.Path(".").glob("20news-bydate-*/*/*")
]
print(f"{len(contenus)} textes récupérés")

In [ ]:
print(contenus[1])

In [ ]:
# Retire l'en-tête du message, les lignes qui annoncent une citation
# (« … writes: ») et la ponctuation, puis découpe en mots
def pretraiter(contenu: str) -> list[str]:
  debut = contenu.find("\n\n")  # L'en-tête finit à la première ligne vide
  if debut > 0:
    contenu = contenu[debut + 2 :]
  mots_par_ligne = (
    re.sub(r'[\!"#$%&\*+,-./:;<=>?@^_`()|~=]', " ", ligne).split()
    for ligne in contenu.splitlines()
    if not ligne.endswith("writes:")
  )
  return list(itertools.chain.from_iterable(mots_par_ligne))


textes = [pretraiter(contenu) for contenu in contenus]

In [ ]:
print(textes[0])

## Détection d'expressions fréquentes (collocations)

Certaines paires de mots sont très courantes (*New York*) : mieux vaut les traiter comme un seul « mot » du vocabulaire, `New_York`.

La cellule ci-dessous retient les paires de mots qui se suivent bien plus souvent que le hasard ne le prévoit, avec le score de l'article de word2vec ([Mikolov et al., 2013](https://arxiv.org/abs/1310.4546)) : $\frac{\text{compte}(ab) - \delta}{\text{compte}(a) \times \text{compte}(b)}$, où $\delta$ écarte les paires trop rares ; multiplié par le nombre de mots du corpus, ce score dit combien de fois plus souvent que le hasard la paire apparaît. Sans analyse grammaticale, on y trouve des noms propres (`New_York`, `United_States`), mais aussi des paires sans intérêt comme `at_least`. Elle prend une vingtaine de secondes.

In [ ]:
# Deux mots qui se suivent bien plus souvent que le hasard ne le voudrait forment
# une expression : score de Mikolov et al. (2013), (compte(ab) - δ) / (compte(a) × compte(b))
comptes_mots = collections.Counter(mot for texte in textes for mot in texte)
comptes_paires = collections.Counter(
  p for texte in textes for p in zip(texte, texte[1:])
)
total = sum(comptes_mots.values())  # × total : le score devient un rapport au hasard
expressions = {
  (a, b)
  for (a, b), compte in comptes_paires.items()
  if (compte - 5) * total / (comptes_mots[a] * comptes_mots[b]) > 100
}


def grouper(texte):
  # Remplace chaque paire d'une expression par un seul mot, a_b
  groupe = []
  for mot in texte:
    if groupe and (groupe[-1], mot) in expressions:
      groupe[-1] = groupe[-1] + "_" + mot
    else:
      groupe.append(mot)
  return groupe


textes_groupes = [grouper(texte) for texte in textes]
print(textes_groupes[0])

## 4. Entraîner un Skip-Gram

Skip-Gram prédit les voisins d'un mot à partir du mot central (la diapositive « Apprentissage (CBOW et Skip-Gram) »). Il apprend deux vecteurs par mot : celui du mot au centre d'une fenêtre, et celui du mot comme voisin ; le score d'un couple (centre, voisin) est le produit scalaire de ces deux vecteurs. Plutôt que de prédire le voisin parmi tous les mots du vocabulaire, l'échantillonnage négatif apprend à distinguer le vrai voisin de quelques mots tirés au hasard, les *négatifs*.

La cellule ci-dessous prépare les données : le vocabulaire (les mots vus au moins 5 fois, du plus fréquent au plus rare), le corpus en une seule suite de numéros de mots, et un sous-échantillonnage, qui retire le plus souvent les mots très fréquents (*the*, *of*…). `torch.manual_seed(0)` fixe le point de départ du hasard, pour le sous-échantillonnage comme pour l'entraînement qui suit.

In [ ]:
# Vocabulaire : les mots vus au moins 5 fois, du plus fréquent au plus rare
comptes = collections.Counter(mot for texte in textes_groupes for mot in texte)
vocabulaire = [mot for mot, compte in comptes.most_common() if compte >= 5]
numeros = {mot: numero for numero, mot in enumerate(vocabulaire)}
frequences = torch.tensor([comptes[mot] for mot in vocabulaire], dtype=torch.float)
# Le corpus : une seule suite de numéros de mots
corpus = torch.tensor(
  [numeros[mot] for texte in textes_groupes for mot in texte if mot in numeros]
)
# Sous-échantillonnage : un mot très fréquent (the, of…) est retiré le plus souvent
torch.manual_seed(0)
garde = torch.rand(len(corpus)) < torch.sqrt(
  1e-3 * frequences.sum() / frequences[corpus]
)
corpus = corpus[garde]
print(len(vocabulaire), "mots,", len(corpus), "occurrences gardées")

### 4.1. Le modèle et sa perte

Complétez la méthode `forward` de `SkipGram` et la fonction `perte`. `SkipGram` a deux tables de vecteurs, `centres` et `contextes`, déjà créées : `forward(centre, contexte)` rend le score de chaque couple, le produit scalaire du vecteur de `centre` dans `centres` et de celui de `contexte` dans `contextes`. `perte(skipgram, centre, contexte, negatifs)` reçoit un lot de centres et de vrais voisins (des numéros de mots, chacun de forme `(lot,)`) et, pour chaque centre, `k` négatifs (de forme `(lot, k)`). Elle rend la moyenne sur le lot de la perte de l'échantillonnage négatif, pour un centre $c$, son vrai voisin $o$ et ses négatifs $n_1, \dots, n_k$ :

$$-\log \sigma(s(c, o)) - \sum_{i=1}^{k} \log \sigma(-s(c, n_i))$$

où $s$ est le score et $\sigma$ la sigmoïde. Elle est faible quand le vrai voisin a un grand score et les négatifs un petit.

**Documentation utile**

- [`torch.nn.Embedding`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)
- [`torch.nn.functional.logsigmoid`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.logsigmoid.html)
- [`torch.Tensor.sum`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.sum.html)
- [`torch.Tensor.mean`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.mean.html)
- [La diffusion (*broadcasting*)](https://docs.pytorch.org/docs/stable/notes/broadcasting.html)

In [ ]:
class SkipGram(torch.nn.Module):
  def __init__(self, taille_vocabulaire, dimension):
    super().__init__()
    # Deux tables : le vecteur d'un mot au centre, et son vecteur comme voisin
    self.centres = torch.nn.Embedding(taille_vocabulaire, dimension, sparse=True)
    self.contextes = torch.nn.Embedding(taille_vocabulaire, dimension, sparse=True)
    # L'initialisation de word2vec : des centres petits, des contextes nuls
    torch.nn.init.uniform_(self.centres.weight, -0.5 / dimension, 0.5 / dimension)
    torch.nn.init.zeros_(self.contextes.weight)

  def forward(self, centre, contexte):
    # Votre code ici
    ...


def perte(skipgram, centre, contexte, negatifs):
  # Votre code ici
  ...

#### Conseils supplémentaires

`self.centres(centre)` rend les vecteurs des mots de `centre`, de forme `(lot, dimension)` ; le produit terme à terme `*`, puis `.sum(dim=-1)` sur la dernière dimension, donne les produits scalaires. Dans `perte`, `skipgram(centre, contexte)` rend les scores des vrais couples, et `skipgram(centre[:, None], negatifs)` ceux des négatifs, de forme `(lot, k)` : `centre[:, None]`, de forme `(lot, 1)`, se diffuse contre `negatifs`. `torch.nn.functional.logsigmoid(x)` calcule $\log \sigma(x)$ ; la somme sur les négatifs se fait sur `dim=1`, la moyenne sur tout le lot. Pour vérifier : avec des contextes nuls, comme à l'initialisation, chaque score vaut 0, et la perte $(k + 1) \log 2$, soit 4,159 pour 5 négatifs.

```python
class SkipGram(torch.nn.Module):
  # __init__ comme dans la cellule d'exercice

  def forward(self, centre, contexte):
    return (self.centres(centre) * ...).sum(dim=...)


def perte(skipgram, centre, contexte, negatifs):
  vrai = torch.nn.functional.logsigmoid(skipgram(centre, contexte))
  faux = torch.nn.functional.logsigmoid(...).sum(dim=1)
  return ...


negatifs = torch.tensor([[2, 3, 4, 5, 6]])
print(perte(SkipGram(10, 4), torch.tensor([0]), torch.tensor([1]), negatifs))  # 4.159
```

#### Solution

In [ ]:
class SkipGram(torch.nn.Module):
  def __init__(self, taille_vocabulaire, dimension):
    super().__init__()
    # Deux tables : le vecteur d'un mot au centre, et son vecteur comme voisin
    self.centres = torch.nn.Embedding(taille_vocabulaire, dimension, sparse=True)
    self.contextes = torch.nn.Embedding(taille_vocabulaire, dimension, sparse=True)
    # L'initialisation de word2vec : des centres petits, des contextes nuls
    torch.nn.init.uniform_(self.centres.weight, -0.5 / dimension, 0.5 / dimension)
    torch.nn.init.zeros_(self.contextes.weight)

  def forward(self, centre, contexte):
    # Score d'un couple : le produit scalaire de ses deux vecteurs
    return (self.centres(centre) * self.contextes(contexte)).sum(dim=-1)


def perte(skipgram, centre, contexte, negatifs):
  # -log σ(score du vrai voisin) - Σ log σ(-score d'un négatif), en moyenne sur le lot
  vrai = torch.nn.functional.logsigmoid(skipgram(centre, contexte))
  faux = torch.nn.functional.logsigmoid(-skipgram(centre[:, None], negatifs)).sum(dim=1)
  return -(vrai + faux).mean()

### 4.2. L'entraînement

La boucle ci-dessous entraîne le Skip-Gram sur `corpus`. À chaque pas, elle prend un lot de 4 096 positions du corpus, un voisin au hasard dans la fenêtre de chacune, et 5 négatifs par centre, tirés selon la fréquence des mots élevée à la puissance 0,75 ; `SparseAdam` ne met à jour que les lignes des mots du lot. Les trois époques prennent de trois à six minutes sur deux cœurs, selon leur charge : lancez-la, et lisez la suite en attendant. La perte affichée baisse d'une époque à l'autre, de 2,8 à 2,4 environ.

Les vecteurs appris, ceux de la table `centres`, vont ensuite dans un `Embeddings`, comme ceux de spaCy : `modele_forums`.

In [ ]:
DIMENSION, FENETRE, NEGATIFS, LOT = 100, 5, 5, 4096
EPOQUES = 3

skipgram = SkipGram(len(vocabulaire), DIMENSION)
optimiseur = torch.optim.SparseAdam(skipgram.parameters(), lr=0.003)
tirage_negatifs = frequences**0.75  # un peu plus de chances aux mots rares

debut = time.time()
for epoque in range(EPOQUES):
  total = 0.0
  lots = torch.randperm(len(corpus) - 2 * FENETRE).add(FENETRE).split(LOT)
  for positions in lots:
    # Un voisin tiré dans la fenêtre : à 1..FENETRE mots, à gauche ou à droite
    decalage = torch.randint(1, FENETRE + 1, positions.shape)
    decalage = decalage * (2 * torch.randint(0, 2, positions.shape) - 1)
    centre, contexte = corpus[positions], corpus[positions + decalage]
    negatifs = torch.multinomial(
      tirage_negatifs, len(positions) * NEGATIFS, replacement=True
    )
    valeur = perte(skipgram, centre, contexte, negatifs.view(-1, NEGATIFS))
    optimiseur.zero_grad()
    valeur.backward()
    optimiseur.step()
    total += valeur.item()
  print(
    f"époque {epoque + 1} : perte {total / len(lots):.3f}, {time.time() - debut:.0f} s"
  )

# Les vecteurs appris, dans la même classe que ceux de spaCy
modele_forums = Embeddings(numeros, skipgram.centres.weight.detach().numpy())

### 4.3. Ce qu'apporte un corpus du domaine

Comparez les voisins de `Windows`, de `DOS`, de `dos` et de `New_York` dans `modele_forums` et dans `modele`, les vecteurs de spaCy : qu'apporte un corpus de votre domaine ? Les deux modèles connaissent-ils tous ces mots ? Essayez enfin `analogie(modele_forums, "man", "woman", "king")` : pourquoi ce petit corpus échoue-t-il, là où les vecteurs de spaCy trouvaient *reine* pour *homme*, *femme*, *roi* (exercice 2) ?

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`modele_forums` est un `Embeddings`, comme `modele` : mêmes `voisins`, même `in`, et `analogie` lui convient. `New_York` n'existe que dans `modele_forums` : testez `mot in modele` avant `modele[mot]`. Dans `modele_forums`, `Windows_NT` est parmi les voisins de `Windows`, et ceux de `DOS` et de `dos` sont des mots de l'informatique ; dans `modele`, `dos` a pour voisins `cou`, `épaule`, `poignet`.

```python
for mot in ["Windows", "DOS", "dos", "New_York"]:
  print("20 Newsgroups", mot, modele_forums.voisins(modele_forums[mot]))
  if mot in modele:
    print("spaCy", mot, ...)
  else:
    ...

print(analogie(...))
```

#### Solution

In [ ]:
# 20 Newsgroups : Windows, DOS et dos ont pour voisins l'informatique de 1993
# (Windows_NT, Win_3…), et New_York des voisins moins nets, qui changent avec la
# graine ; spaCy : les versions récentes de Windows, et dos, le dos du corps
for mot in ["Windows", "DOS", "dos", "New_York"]:
  print("20 Newsgroups", mot, modele_forums.voisins(modele_forums[mot]))
  if mot in modele:
    print("spaCy", mot, modele.voisins(modele[mot]))
  else:
    print("spaCy", mot, "absent")

# Moins de 5 millions de mots, peu de rois et de reines : l'analogie échoue
print(analogie(modele_forums, "man", "woman", "king"))